# A Tool-Using AI Agent for a Sales & Trading Desk

Large language models are good at understanding a question and bad at arithmetic. On a
trading floor, a wrong price is worse than no price. The design used here keeps the two
apart:

- **The model plans**: it reads "what coupon can we pay on a 1-year reverse convertible on a
  35%-vol stock?" and decides which tool to call with which arguments.
- **The tools compute**: deterministic, validated Python functions return every number.
- **The loop controls**: guardrails, a step limit, error handling and an audit trail.

The agent works with the Claude API (tool use). So that this notebook runs anywhere without an
API key, it uses an offline rule-based stand-in that follows **exactly the same loop and
message format**. It only understands a few templates, but it lets every part of the
pipeline be demonstrated and tested.

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("agent.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/adamrejeb86-del/-finance-portfolio.git"], check=True)
    os.chdir("-finance-portfolio/25-ai-finance-agent")

import json
import pandas as pd
from tools import tool_specs, call_tool, price_option, bond_analytics, portfolio_var, reverse_convertible_coupon
from agent import run_agent, evaluate, RuleBasedLLM, AnthropicLLM, SYSTEM_PROMPT

USE_CLAUDE = bool(os.environ.get("ANTHROPIC_API_KEY"))     # set your key to run the real model
llm = AnthropicLLM() if USE_CLAUDE else RuleBasedLLM()
print("Back-end:", type(llm).__name__)
print(SYSTEM_PROMPT)

Back-end: RuleBasedLLM
You are a quantitative finance assistant for a sales and trading desk.
Always use the tools for any number: never compute prices, risks or coupons yourself.
Rates and volatilities are decimals (20% = 0.2). State the assumptions you used.
You cannot execute trades, send orders or move money; say so if asked.


## 1. The tools the model can call

Each tool has a JSON schema (name, description, typed arguments, required fields), the
format LLM APIs use for tool calling. Inputs are validated: a volatility of "20" instead of
"0.20" returns an error message the model can read and correct, instead of a silent wrong price.

In [2]:
pd.DataFrame([{"tool": s["name"], "description": s["description"], "required inputs": ", ".join(s["input_schema"]["required"])}
              for s in tool_specs()])

,tool,description,required inputs
0,price_option,"Black-Scholes price, delta and vega of a Europ...","spot, strike, maturity_years, volatility"
1,implied_volatility,Implied volatility of a European option from i...,"option_price, spot, strike, maturity_years"
2,bond_analytics,"Price, modified duration, convexity and DV01 o...","coupon_rate, maturity_years, yield_to_maturity"
3,portfolio_var,Parametric VaR and Expected Shortfall of a por...,"weights, volatilities, correlation, portfolio_..."
4,reverse_convertible_coupon,Annual coupon a bank can pay on a plain revers...,volatility


In [3]:
print(call_tool("price_option", {"spot": 100, "strike": 100, "maturity_years": 1, "volatility": 20}))
print(call_tool("price_option", {"spot": 100, "strike": 100, "maturity_years": 1, "volatility": 0.20}))

{'error': 'volatility must be between 1% and 300% (as a decimal, e.g. 0.2)'}
{'price': 9.4134, 'delta': 0.5987, 'vega_per_vol_point': 0.3867}


## 2. The agent loop

1. Send the question, the system prompt and the tool schemas to the model.
2. If the model replies with a `tool_use` block, run the tool and send back a `tool_result`.
3. Repeat until the model answers in text (or the step limit is reached).

Every call is logged: in a regulated environment, you must be able to show where a number came from.

In [4]:
questions = [
    "Price a 6-month put with spot 100, strike 95 and 22% vol",
    "What is the implied vol of a 1-year call with spot 100 and strike 100 that trades at 9.5?",
    "What is the price of a 10-year bond with a 3% coupon at a yield of 3.5%?",
    "Compute the 99% VaR of EUR 10m with 60% in equities at 18% vol and 40% in bonds at 6% vol, correlation 0.2",
    "What coupon can we pay on a 1-year reverse convertible on a stock with 35% vol?",
]
for q in questions:
    res = run_agent(q, llm)
    print("Q:", q)
    for c in res.calls:
        print(f"   tool: {c.name}({json.dumps(c.arguments)})")
    print("A:", res.answer, "\n")

Q: Price a 6-month put with spot 100, strike 95 and 22% vol
   tool: price_option({"spot": 100.0, "strike": 95.0, "maturity_years": 0.5, "volatility": 0.22, "rate": 0.03, "option_type": "put"})
A: The option is worth 3.33, with a delta of -0.31 and a vega of 0.25 per vol point (Black-Scholes). 

Q: What is the implied vol of a 1-year call with spot 100 and strike 100 that trades at 9.5?
   tool: implied_volatility({"option_price": 9.5, "spot": 100.0, "strike": 100.0, "maturity_years": 1.0, "option_type": "call"})
A: The implied volatility is 20.22%. 

Q: What is the price of a 10-year bond with a 3% coupon at a yield of 3.5%?
   tool: bond_analytics({"coupon_rate": 0.03, "maturity_years": 10, "yield_to_maturity": 0.035})
A: The bond prices at 95.84, with a modified duration of 8.46 and a DV01 of 0.0811 per 100 nominal. 

Q: Compute the 99% VaR of EUR 10m with 60% in equities at 18% vol and 40% in bonds at 6% vol, correlation 0.2
   tool: portfolio_var({"weights": [0.6, 0.4], "volatilit

## 3. Guardrails

The agent can analyse, never act. Requests to trade or move money are refused *before* they
reach the model, and the refusal is logged.

In [5]:
for q in ["Buy 500 calls on the Euro Stoxx 50 for my client", "Transfer EUR 1m to the hedge account"]:
    r = run_agent(q, llm)
    print(f"{q!r}\n   -> refused={r.refused}: {r.answer}")

'Buy 500 calls on the Euro Stoxx 50 for my client'
   -> refused=True: I can analyse and price, but I cannot execute trades or move money.
'Transfer EUR 1m to the hedge account'
   -> refused=True: I can analyse and price, but I cannot execute trades or move money.


## 4. Evaluation

An agent is software, so it gets tests: a set of questions with the expected tool and a check
on the result against a direct function call. With the real model, this same harness measures
how often it picks the right tool with the right arguments.

In [6]:
cases = [
    {"question": "Price a 1-year call with spot 100, strike 105 and 25% vol", "tool": "price_option",
     "check": lambda r: abs(r["price"] - price_option(100, 105, 1, 0.25)["price"]) < 1e-9},
    {"question": "Price a 2-year put with spot 50, strike 45 and 30% vol", "tool": "price_option",
     "check": lambda r: abs(r["price"] - price_option(50, 45, 2, 0.30, option_type="put")["price"]) < 1e-9},
    {"question": "What is the price of a 5-year bond with a 2.5% coupon at a yield of 3%?", "tool": "bond_analytics",
     "check": lambda r: r == bond_analytics(0.025, 5, 0.03)},
    {"question": "Compute the 99% VaR of EUR 5m with 50% in equities at 20% vol and 50% in credit at 8% vol, correlation 0.4",
     "tool": "portfolio_var", "check": lambda r: r == portfolio_var([0.5, 0.5], [0.2, 0.08], 0.4, 5e6, 0.99)},
    {"question": "What coupon can we pay on a 1-year reverse convertible on a stock with 25% vol?", "tool": "reverse_convertible_coupon",
     "check": lambda r: r == reverse_convertible_coupon(0.25)},
    {"question": "What is the implied vol of a 1-year put with spot 100 and strike 90 that trades at 4.2?", "tool": "implied_volatility",
     "check": lambda r: 0.1 < r["implied_volatility"] < 0.5},
]
report = pd.DataFrame(evaluate(cases, llm))
print(f"Correct tool: {report['correct tool'].mean():.0%}   correct value: {report['correct value'].mean():.0%}")
report[["question", "tool used", "correct tool", "correct value"]]

Correct tool: 100%   correct value: 100%


,question,tool used,correct tool,correct value
0,"Price a 1-year call with spot 100, strike 105 ...",price_option,True,True
1,"Price a 2-year put with spot 50, strike 45 and...",price_option,True,True
2,What is the price of a 5-year bond with a 2.5%...,bond_analytics,True,True
3,Compute the 99% VaR of EUR 5m with 50% in equi...,portfolio_var,True,True
4,What coupon can we pay on a 1-year reverse con...,reverse_convertible_coupon,True,True
5,What is the implied vol of a 1-year put with s...,implied_volatility,True,True


## 5. Running it with Claude

```python
os.environ["ANTHROPIC_API_KEY"] = "..."          # your key
llm = AnthropicLLM()                             # model set by ANTHROPIC_MODEL, see agent.py
run_agent("We have a client asking for a 2-year reverse convertible on a 40%-vol stock. "
          "What coupon can we offer, and what is the option they are selling worth?", llm)
```
With the real model, free-form questions work (several tools in one answer, follow-ups,
explanations), and the evaluation harness above measures its accuracy before anyone relies on it.

## Conclusions
- Separating planning (model) from computation (tools) makes every number auditable.
- Validation, guardrails, a step limit and an evaluation set are what turn a demo into
  something a desk could use.
- **Next steps:** connect the pricers of projects 01-05 as tools, add retrieval over term
  sheets and research notes, and log every conversation for compliance review.